# GW assignment 3

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import astropy.units as u
import astropy.cosmology as cosmo
from astropy.cosmology import FlatwCDM
cosmol = FlatwCDM(H0=67.9, Om0=0.3065, w0=-1)

import numpy as np
import matplotlib.pyplot as plt

#IMPORTANT: I am importing gwpy's TimeSeries as GWPYTimeSeries to avoid name clashes with PyCBC's TimeSeries
#Make sure your code uses the correct TimeSeries class!
from gwpy.timeseries import TimeSeries as GWPYTimeSeries
from pycbc.filter import resample_to_delta_t, highpass
from pycbc.psd import interpolate, inverse_spectrum_truncation, aLIGOZeroDetHighPower
from pycbc.types import TimeSeries, FrequencySeries
from pycbc.filter import matched_filter
from pycbc.noise import noise_from_psd
from pycbc.detector import Detector
from pycbc.waveform import get_td_waveform
import bilby

### Q1a: Generate the GW waveform of a merger between two 20 solar mass black holes at a luminosity distance of 1000 Mpc. Inject the GW signal into Gaussian noise to obtain the strain. Plot the strain, with the injected waveform overlaid. (hint: see workshop tutorial 2.1 for help aligning the waveform properly)

In [ ]:
#you should use these parameters for your waveform and noise
sample_rate = 2048 # samples per second
duration = 32 # seconds
f_lower = 15.0 # lowest frequency in Hz

delta_t = 1.0 / sample_rate
approximant = "IMRPhenomD"

#this is the PSD you should use for the noise
psd = aLIGOZeroDetHighPower(sample_rate, duration, f_lower)
#Your code here



### Q1b: plot the Q transform of the strain, such that the merger is visible. Plot the transform with different q ranges. What is the effect of changing the the qrange parameter on the Q transform?

In [ ]:
#Your code here


### Q1c: Pick a gravitational wave signal from the GWTC3 catalogue: https://link.aps.org/doi/10.1103/PhysRevX.13.041039 (see table IV, or the [GWOSC webpage](https://gwosc.org/eventapi/html/query/show?release=GWTC-3-confident)) with a network SNR above 15 and fetch the strain for both detectors using gwpy. Plot its Q transform for each detector. (note the function to fetch the data is aliased to GWPYTimeSeries.fetch_open_data to avoid confusion with PyCBC's TimeSeries )

In [ ]:
from pycbc.catalog import Merger

#Your code here


### Q1d: Use gwpy to download the Hanford and Livingston data 8 seconds either side of the gps time given below. Plot the Q transform of the data for each detector. What do you observe in the Q transform?

In [ ]:
duration = 16
gps_time = 1244875090.25

#Your code here

### Q2: Use matched filtering to plot the SNR time series for the signal you generated in Q1a. What is the SNR of the signal? Note: You should use the signal as the template for matched filtering, and don't need to project the waveform onto the detectors.

In [ ]:
#Your code here

In [ ]:
#need to crop to get rid of filter ringing on the edges
crop = 1
snr.data[:crop*sample_rate] = 0
snr.data[-crop*sample_rate:] = 0

#taking the absolute value of the snr as we don't care about phase
snr = np.abs(snr)

times = np.linspace(0, duration, len(snr))
plt.plot(times, snr)

plt.ylabel("SNR")
plt.xlabel("time (s)")

### Q3. Use matched filtering to identify multiple signals in a section of pregenerated noise. For each signal, find the time of the merger, and the approximate chirp mass of the signal. For each signal, plot the SNR time series 1 second either side of the merger for both of the detectors, and state the network SNR you recover the signal with.

Additional information:
There are 5 signals to find, and all masses are a multiple of 5 and are in the range 5 - 50 solar masses.
None of the signals are injected in the first or last 100 seconds of the data. Why might this be important? Look into the process of discrete time to frequency domain matched filtering for information on this.

The noise and PSD are provided in the files Q3_strain_data.npz and Q3_psd_data.npz respectively. The code block below will help with loading the data.

In [ ]:
import numpy as np

In [ ]:
start_time = 1240210479
sample_rate = 4096
duration = 1024
f_lower = 30

delta_t = 1.0 / sample_rate
delta_f = 1.0 /duration
ifos = ['H1', 'L1']

strain = np.load("Q3_strain_data.npz")
psds = np.load("Q3_psd_data.npz")

#converting strain and PSD to PyCBC TimeSeries and FrequencySeries
strain = {ifo: TimeSeries(strain[ifo], delta_t=delta_t) for ifo in ifos}
psds = {ifo: FrequencySeries(psds[ifo], delta_f=delta_f) for ifo in ifos}

In [ ]:
#An example of how to use the Hanford data from the loaded files
plt.plot(strain['H1'].sample_times, strain['H1'], label='H1')

### Q4: Now you are going to investigate the relationship between chirp mass and sky localisation area. 

Sky localisation is the estimation of the right ascension and declination parameters of a signal.

As a warmup, check that you can run the below code (not assessed). You should see that when you run bilby.run_sampler, a posterior is produced, with right ascension and declination values similar to the values in the injection_parameters dictionary. Note that with my computer this code takes ~2 minutes to run.

In [ ]:
#Example taken from: https://github.com/bilby-dev/bilby/blob/main/examples/gw_examples/injection_examples/plot_skymap.py
import bilby
from bilby.core.utils.random import seed

# Sets seed of bilby's generator "rng" to "123" to ensure reproducibility
seed(123)

duration = 4
sampling_frequency = 1024
outdir = "outdir"
label = "plot_skymap"
injection_parameters = dict(
    mass_1=36.0,
    mass_2=29.0,
    a_1=0.4,
    a_2=0.3,
    tilt_1=0.5,
    tilt_2=1.0,
    phi_12=1.7,
    phi_jl=0.3,
    luminosity_distance=4000.0,
    theta_jn=0.4,
    psi=2.659,
    phase=1.3,
    geocent_time=1126259642.413,
    ra=1.375,
    dec=-0.2108,
)

waveform_arguments = dict(waveform_approximant="IMRPhenomXP", reference_frequency=50.0)

waveform_generator = bilby.gw.WaveformGenerator(
    duration=duration,
    sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameters=injection_parameters,
    waveform_arguments=waveform_arguments,
)

ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=sampling_frequency,
    duration=duration,
    start_time=injection_parameters["geocent_time"] - 2,
)
ifos.inject_signal(
    waveform_generator=waveform_generator, parameters=injection_parameters
)

priors = bilby.gw.prior.BBHPriorDict()
for key in [
    "a_1",
    "a_2",
    "tilt_1",
    "tilt_2",
    "phi_12",
    "phi_jl",
    "psi",
    "mass_1",
    "mass_2",
    "phase",
    "geocent_time",
    "theta_jn",
]:
    priors[key] = injection_parameters[key]
del priors["chirp_mass"], priors["mass_ratio"]

likelihood = bilby.gw.GravitationalWaveTransient(
    interferometers=ifos, waveform_generator=waveform_generator
)

result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    sampler="nestle",
    npoints=250,
    injection_parameters=injection_parameters,
    outdir=outdir,
    label=label,
    result_class=bilby.gw.result.CBCResult, npool = 4
)


In [ ]:

# make some plots of the outputs
result.plot_corner()


In [ ]:
result.plot_skymap(maxpts=500)


#### Q4a: Assuming all other parameters are fixed (including SNR), how would you expect the sky localization area to change if the chirp mass was increased or decreased? Give physical arguments. (Hint: consider the dependence of the GW localization on frequencies. Reference: https://journals.aps.org/prd/abstract/10.1103/PhysRevD.81.082001).

#### Q4b: Now you will test your expectations from Q4a. Choose a sky location (You should choose a right ascension value between 0 and 2 $\pi$, and a declination value between $-\pi/2$ and $\pi/2$) and two component mass pairs. The component mass values should be significantly different e.g. a factor of 6 or more apart, and shouldn't be any lower than 5 solar masses (e.g. a signal with masses 5,5 and a signal with masses 40,40). 

#### Use the initialise_bilby function below to estimate the optimal SNR of each signal for both the H1 and L1 detectors (it will print these values in its output). For each signal, rerun initialise_bilby with different luminosity distance values until the optimal SNRs for each signal are close to the same value. You should aim for a relatively low SNR (e.g. less than 7) so that the sky localization areas are large enough to see a difference. For example, if for signal 1, your H1 SNR is 6 and your L1 SNR is 7, signal 2 should have an H1 SNR of ~5.9-6.1 and an L1 SNR of ~6.9-7.1.

In [ ]:
import bilby
from bilby.core.utils.random import seed


In [ ]:
duration = 32.0
sampling_frequency = 1024.0
minimum_frequency = 20

mergertime = 1126259642.413

psd_duration = 1024
roll_off = 0.4

waveform_arguments = dict(
    waveform_approximant="IMRPhenomPv2",
    reference_frequency=50.0,
    minimum_frequency=minimum_frequency,
)

waveform_generator = bilby.gw.WaveformGenerator(
    duration=duration,
    sampling_frequency=sampling_frequency,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_arguments,
)

In [ ]:

def initialise_bilby(mass1, mass2, distance, right_ascension, declination):
	injection_parameters = dict(
		mass_1=mass1,
		mass_2=mass2,
		a_1=0,
		a_2=0,
		tilt_1=0,
		tilt_2=0,
		phi_12=0,
		phi_jl=0,
		luminosity_distance=distance,
		theta_jn=0,
		psi=0,
		phase=0,
		geocent_time=mergertime,
		ra=right_ascension,
		dec=declination,
	)


	ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
	ifos.set_strain_data_from_power_spectral_densities(
		sampling_frequency=sampling_frequency,
		duration=duration,
		start_time=mergertime - 20,
	)
	ifos.inject_signal(
		waveform_generator=waveform_generator, parameters=injection_parameters
	)

	priors = bilby.gw.prior.BBHPriorDict()

	del(priors['chirp_mass'])
	del(priors['mass_ratio'])
	#we don't modify the ra and dec distributions as those are set correctly by BBHPriorDict
	#note that most parameters are fixed to the injection values
	priors['mass_1'] = mass1
	priors['mass_2'] = mass2
	priors["luminosity_distance"] = distance
	priors["geocent_time"] = mergertime
	priors["psi"] = 0
	priors["theta_jn"] = 0
	priors["phase"] = 0
	priors["a_1"] = 0
	priors["a_2"] = 0
	priors["tilt_1"] = 0
	priors["tilt_2"] = 0
	priors["phi_12"] = 0
	priors["phi_jl"] = 0

	likelihood = bilby.gw.GravitationalWaveTransient(
		interferometers=ifos, waveform_generator=waveform_generator,priors=priors
	)

	print(priors)
	return likelihood, priors, injection_parameters


In [ ]:
#Your code here

#### Q4c: With the component mass and luminosity distance values you found in Q4b, choose a sky location and then use Bilby to estimate the sky locations of your signals. You should choose a right ascension value between 0 and 2 $\pi$, and a declination value between $-\pi/2$ and $\pi/2$ . Plot the skymap for each signal using the code below. How does your result compare to your expectations from the physical arguments you gave above? 


Note: you should check in the output from running the initialise_bilby function that the Optimal SNRs for each signal are close to the same value. If not, tweak your luminosity distances until they are close. (eg if for signal 1, your H1 SNR is 6 and your L1 SNR is 7, signal 2 should have an H1 SNR of ~5.9-6.1 and an L1 SNR of ~6.9-7.1.).


In [ ]:
#small test run. For the real run increase npoints to 1000 or more and reduce dlogz to 0.1
#note that the full run could take several hours depending on your injection parameters and computer.
#you may also want to try the 'nestle' sampler instead of 'dynesty' if your computer is slow
likelihood, priors, injection_parameters = #your code here
result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    sampler="dynesty",
    npoints=250,
    injection_parameters=injection_parameters, dlogz=3,
    outdir='short_test', label="test", result_class=bilby.gw.result.CBCResult, npool=4
)


In [ ]:

# make some plots of the outputs
result.plot_corner()


In [ ]:
result.plot_skymap(maxpts=100)

#### Q4d: Compare your localisation areas with the GWTC-3 catalogue. You should find that your areas are significantly smaller compared to signals with similar SNR. Why is this? To investigate, choose some of the prior values to be uniform over a range rather than fixed, and rerun the Bilby analysis. Note that if you choose to vary the distance, phase or time, you should specify distance_marginalization=True, phase_marginalisation = True, etc, in the likelihood function.

Note: to unconstrain a parameter, you should set its value to a bilby prior object. You also need to pass the prior object the name of the parameter. For example, to make the luminosity distance uniform in the source frame between 100 and 5000 Mpc, you would use:
```python
priors["luminosity_distance"] = bilby.gw.prior.UniformSourceFrame(100, 5000, name="luminosity_distance")
```
